# Measure It — Google Colab training and evaluation

Before starting: in Colab choose **Runtime → Change runtime type → T4 GPU** (or any available GPU). On your PC, make a ZIP of the project containing `dataset/`, `models/`, `scripts/`, `calibration/`, `inference/`, and the project `.py` files. Include reviewed masks for train and validation; add test masks before final evaluation. Do not include `.venv/` or `.git/`. Upload that ZIP below.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
print('Uploaded:', zip_name)

In [ ]:
import os, pathlib, zipfile
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('/content/measure-it')
candidates = list(pathlib.Path('/content/measure-it').rglob('models/train.py'))
assert candidates, 'ZIP must contain models/train.py'
PROJECT = candidates[0].parent.parent
os.chdir(PROJECT)
print('Project:', PROJECT)
import torch, torchvision
print('Python:', __import__('sys').version)
print('Torch:', torch.__version__, '| TorchVision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import cv2, PIL
print('OpenCV:', cv2.__version__, '| Pillow:', PIL.__version__)
!python scripts/check_labels.py

In [ ]:
!python models/train.py --epochs 10 --seed 42

In [ ]:
# Run only after training is finished and the model/threshold are frozen.
!python models/evaluate.py --weights models/maskrcnn.pt

In [ ]:
from google.colab import files
files.download('models/maskrcnn.pt')
files.download('models/training_history.json')
files.download('measurement/test_evaluation/metrics.json')
files.download('measurement/test_evaluation/per_image.csv')